# playground_LN — what the final LayerNorm (gamma, beta) is doing

Final LN linearized as `LN(x) = [gamma/sqrt(sig^2+eps)] (.) x  -  [mu*gamma/sqrt(sig^2+eps) - beta]`
= **multiplicative block** - **subtracted block**, then `@ proj` into the shared space.

- **gamma, beta** are single vectors per tower -> interpret directly (nearest texts/images), NOT via SVD.
- **subtracted block** = `a*gamma - beta`, `a = mu/sqrt(sig^2+eps)` -> lives in span{gamma,beta} (rank<=2; rank-1 after centering).
- **multiplicative block** = `M + a*gamma - beta` (M = final embedding) -> SVD gives representation PCs + the gamma coupling.

Stage 0 (setup, copied verbatim from playground_all) -> A (gamma/beta) -> C (cross-encoder) run now. Stage B needs per-sample mu,sigma (one forward pass, done in-notebook).

## Stage 0 — setup (from playground_all cells 0,1,2,7,8)

In [1]:
## Imports
import numpy as np
import torch
import json
import os
from tabulate import tabulate
from PIL import Image
from torch.nn import functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.datasets import ImageNet, CIFAR10, CIFAR100, ImageFolder
from utils.misc.misc import accuracy, accuracy_correct
from utils.scripts.algorithms_text_explanations import svd_data_approx
from utils.scripts.algorithms_text_explanations_funcs import *
from utils.models.factory import create_model_and_transforms, get_tokenizer
from utils.models.prs_hook import hook_prs_logger
from utils.models.prs_hook_text import hook_prs_logger_text
from utils.misc.visualization import visualization_preprocess
from utils.datasets_constants.imagenet_classes import imagenet_classes
from utils.datasets_constants.cifar_10_classes import cifar_10_classes
from utils.datasets_constants.cub_classes import cub_classes, waterbird_classes
from utils.datasets.dataset_helpers import dataset_subset
from utils.datasets.binary_waterbirds import BinaryWaterbirds


/home/virgilio/lib/minconda3/envs/MT/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
## Parameters
device = 'cpu'
model_name = 'ViT-B-32'        # 'ViT-H-14', 'ViT-L-14', 'ViT-B-16', 'ViT-B-32'
seed = 0
num_last_layers_ = 11            # how many of the LAST layers get a full PC decomposition, for BOTH encoders
algorithm = "svd_data_approx"

# Dataset the vision is run on (both for the activations / visualization)
dataset_image_name = "imagenet"      
subset_dim = 10 
tot_samples_per_class = 10 # nr samples per class
path = './datasets/'

# Dataset the text encoder is run to produce activations
dataset = "imagenet_descriptions_personal" 
native_per_class = 10           # sentences per class in "dataset"
sentences_per_class = 1         # keep the LAST k per class; 1 -> only the class-name sentence

# Shared: labels used to text-label PCs of BOTH encoders
dataset_text_name = "top_1500_nouns_5_sentences_imagenet_clean"

cache_dir = "../cache"

if model_name == "ViT-H-14":
    pretrained = "laion2B-s32B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14":
    pretrained = "laion2B-s32B-b82K"; precision = "fp32"
elif model_name == "ViT-B-16":
    pretrained = "laion2B-s34B-b88K"; precision = "fp32"
elif model_name == "ViT-B-32":
    pretrained = "laion2B-s34B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14-336":
    pretrained = "openai"; precision = "fp16"


In [3]:
## Loading Model — one CLIP checkpoint, both towers
model, _, preprocess = create_model_and_transforms(
    model_name, pretrained=pretrained, precision=precision, cache_dir=cache_dir)
model.to(device)
model.eval()
tokenizer = get_tokenizer(model_name)

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Number of VISION-encoder layers:", len(model.visual.transformer.resblocks))
print("Number of TEXT-encoder layers:", len(model.transformer.resblocks))


Using local files
Model parameters: 151,277,313
Number of VISION-encoder layers: 12
Number of TEXT-encoder layers: 12


In [4]:
# VISION encoder: components, classifier, labels
attention_dataset_v = f"output_dir/{dataset_image_name}_completeness_{dataset_text_name}_{model_name}_algo_{algorithm}_seed_{seed}.jsonl"

attns_v = torch.tensor(np.load(f"output_dir/{dataset_image_name}_attn_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l, h, d]
mlps_v  = torch.tensor(np.load(f"output_dir/{dataset_image_name}_mlp_{model_name}_seed_{seed}.npy",  mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l+1, d]
classifier_ = torch.tensor(np.load(f"output_dir/{dataset_image_name}_classifier_{model_name}.npy", mmap_mode="r")).to(device, dtype=torch.float32)   # [d, C], class-prompt classifier
labels_v = torch.tensor(np.load(f"output_dir/{dataset_image_name}_labels_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()            # [N], class per image

# Shared probes: label (texts) + image probes, used to interpret heads/PCs of BOTH encoders
# final_embeddings_images is kept FULL: the image-viz cells index it by ImageNet position (ds_vis_.indices).
final_embeddings_images = torch.tensor(np.load(f"output_dir/{dataset_image_name}_embeddings_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)
final_embeddings_texts  = torch.tensor(np.load(f"output_dir/{dataset_text_name}_{model_name}.npy", mmap_mode="r")).to(device, dtype=torch.float32)
with open(f"utils/text_descriptions/{dataset_text_name}.txt", "r") as f:
    texts_str = np.array([i.replace("\n", "") for i in f.readlines()])

ds_ = ImageNet(root=path + "imagenet/", split="val", transform=visualization_preprocess)
classes_ = imagenet_classes
ds_vis_ = dataset_subset(ds_, samples_per_class=subset_dim, tot_samples_per_class=tot_samples_per_class, seed=seed)

# Derived quantities (mirrors playground.ipynb)
no_heads_attentions_v = attns_v.sum(axis=2)                # [N, l, d], summed over heads
nr_layers_v = attns_v.shape[1]
nr_heads_v  = attns_v.shape[2]
last_v = nr_layers_v - num_last_layers_                    # first layer index that got a PC decomposition
current_mean_ablation_per_head_sum_v = torch.mean(no_heads_attentions_v[:, :last_v + 1], axis=0).sum(0)

data_v_full = get_data(attention_dataset_v, -1)                       # all PCs, incl. head=-1 summary line
data_v = get_data(attention_dataset_v, -1, skip_final=True)           # all PCs, per-head only
mean_rank_v = sum(e["rank"] for e in data_v) / len(data_v)
print("VISION  attns", tuple(attns_v.shape), "mlps", tuple(mlps_v.shape), "| mean head rank:", round(mean_rank_v, 2))

VISION  attns (50000, 12, 12, 512) mlps (50000, 13, 512) | mean head rank: 63.61


In [5]:
# TEXT encoder: components, labels
attention_dataset_t = f"output_dir/{dataset}_completeness_text_{dataset_text_name}_{model_name}_algo_{algorithm}_seed_{seed}.jsonl"

attns_t = torch.tensor(np.load(f"output_dir/{dataset}_attn_text_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l, h, d]
mlps_t  = torch.tensor(np.load(f"output_dir/{dataset}_mlp_text_{model_name}_seed_{seed}.npy",  mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l+1, d]

# Decomposed-corpus texts, aligned with attns_t rows (same last-k-per-class subsampling as prepare)
with open(f"utils/text_descriptions/{dataset}.txt", "r") as f:
    _all = [i.replace("\n", "") for i in f.readlines()]
corpus_texts = np.array([l for c in range(len(_all) // native_per_class)
                         for l in _all[(c + 1) * native_per_class - sentences_per_class:(c + 1) * native_per_class]])

# Class labels (row -> ImageNet class) and per-image labels, for text-encoder accuracy
labels_t = torch.tensor(np.load(f"output_dir/{dataset}_labels_text_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()
image_labels_t = torch.tensor(np.load(f"output_dir/{dataset_image_name}_labels_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()

# Exact decomposition target: M_text = sum_l sum_h attn + sum_l mlp  (EOS, projected)
final_embeddings_texts_decomp = attns_t.sum(axis=(1, 2)) + mlps_t.sum(axis=1)  # [N, d]

no_heads_attentions_t = attns_t.sum(axis=2)      # [N, l, d]
nr_layers_t = attns_t.shape[1]
nr_heads_t  = attns_t.shape[2]
last_t = nr_layers_t - num_last_layers_
current_mean_ablation_per_head_sum_t = torch.mean(no_heads_attentions_t[:, :last_t + 1], axis=0).sum(0)

data_t_full = get_data(attention_dataset_t, -1)
data_t = get_data(attention_dataset_t, -1, skip_final=True)
mean_rank_t = sum(e["rank"] for e in data_t) / len(data_t)
print("TEXT    attns", tuple(attns_t.shape), "mlps", tuple(mlps_t.shape), "| mean head rank:", round(mean_rank_t, 2))


TEXT    attns (1000, 12, 8, 512) mlps (1000, 13, 512) | mean head rank: 63.55


## Stage A — gamma, beta in the shared space: where do they push?
`beta@proj` is the constant shift added to every embedding; `gamma@proj` is the amplification direction (plus the more faithful per-dimension view). Nearest probe texts + images, cosine and dot.

In [6]:
# final-LN params -> shared space
proj_v  = model.visual.proj.detach().float()            # [768,512]
gamma_v = model.visual.ln_post.weight.detach().float()  # [768]
beta_v  = model.visual.ln_post.bias.detach().float()    # [768]
proj_t  = model.text_projection.detach().float()        # [512,512]
gamma_t = model.ln_final.weight.detach().float()        # [512]
beta_t  = model.ln_final.bias.detach().float()          # [512]
gamma_v_s, beta_v_s = gamma_v @ proj_v, beta_v @ proj_v
gamma_t_s, beta_t_s = gamma_t @ proj_t, beta_t @ proj_t

txt_bank = final_embeddings_texts                       # [n,512] probe texts, aligned with texts_str
img_bank = final_embeddings_images                      # [Nfull,512]
img_names = (np.array(imagenet_classes)[np.array(ds_.targets)]
             if len(img_bank) == len(ds_.targets) else None)   # guard: only if lengths match

def nearest(vec, bank, names, k=8, metric="cos"):
    v = vec / vec.norm() if metric == "cos" else vec
    b = bank / bank.norm(dim=-1, keepdim=True) if metric == "cos" else bank
    s = b @ v; top = s.topk(k).indices.tolist()
    return [(names[i], round(s[i].item(), 3)) for i in top]

for name, vec in [("VISION beta", beta_v_s), ("VISION gamma", gamma_v_s),
                  ("TEXT beta", beta_t_s), ("TEXT gamma", gamma_t_s)]:
    print(f"\n=== {name} ===")
    print("  texts(cos):", [t for t,_ in nearest(vec, txt_bank, texts_str, 6)])
    print("  texts(dot):", [t for t,_ in nearest(vec, txt_bank, texts_str, 6, 'dot')])
    if img_names is not None:
        print("  imgs (cos):", [t for t,_ in nearest(vec, img_bank, img_names, 6)])



=== VISION beta ===
  texts(cos): ['An image of a version.', 'An image of a category.', 'An image of an item.', 'An image of a unit.', 'An image of an entry.', 'An image with a sleek appearance.']
  texts(dot): ['An image of a version.', 'An image of a category.', 'An image of an item.', 'An image of a unit.', 'An image of an entry.', 'An image with a sleek appearance.']
  imgs (cos): ['slot machine', 'slot machine', 'hard disk drive', 'slot machine', 'apiary', 'solar thermal collector']

=== VISION gamma ===
  texts(cos): ['An image of a version.', 'An image of a category.', 'An image of an item.', 'An image of a unit.', 'An image that is the first of its kind.', 'An image of an entry.']
  texts(dot): ['An image of a version.', 'An image of a category.', 'An image of an item.', 'An image of a unit.', 'An image that is the first of its kind.', 'An image of an entry.']
  imgs (cos): ['window shade', 'upright piano', 'upright piano', 'upright piano', 'microphone', 'plant pot']

=== TEXT

In [7]:
# faithful gamma view: which pre-proj dims gamma most amplifies, and their concept (proj row -> texts)
for name, gamma, proj in [("VISION", gamma_v, proj_v), ("TEXT", gamma_t, proj_t)]:
    print(f"\n{name}: top-|gamma| dims -> nearest texts of that dim's projection row")
    for d in gamma.abs().topk(6).indices.tolist():
        print(f"  dim {d:3d} gamma={gamma[d]:+.2f}:", [t for t,_ in nearest(proj[d], txt_bank, texts_str, 3)])



VISION: top-|gamma| dims -> nearest texts of that dim's projection row
  dim 758 gamma=+1.97: ['An image of a red and yellow machine.', 'An image of orange, red, and yellow foliage.', 'An image of orange and yellow colors.']
  dim 325 gamma=+1.97: ['An image of a white and silver vehicle.', 'An image of a white and tan animal.', 'An image of a white or cream-colored cat.']
  dim 260 gamma=+1.94: ['An image of a green and yellow object.', 'An image of a lime-colored sphere.', 'An image of a green sphere.']
  dim  45 gamma=+1.83: ['An image of sleek modern gadgets like headphones and smartwatches.', 'An image of a foldable device.', 'An image of a gadget with interchangeable lenses.']
  dim 529 gamma=+1.80: ['An image of a person sleeping with thought bubbles above.', 'An image of a guitar pick on a table.', 'An image of a person rolling something.']
  dim  39 gamma=+1.78: ['An image of a brownish-grey object.', 'An image of a beige and brown structure.', 'An image of a faded photograph

## Stage C — cross-encoder cosine of the LN blocks (shared space)
Are the two towers' LN offsets/scales pointing the same way? And is `beta` the modality-gap direction?

In [8]:
def cd(a, b):
    return (a @ b / (a.norm() * b.norm())).item(), (a @ b).item()

print("cross-encoder (shared space):")
print("  gamma_v <-> gamma_t : cos=%+.3f  dot=%+.3f" % cd(gamma_v_s, gamma_t_s))
print("  beta_v  <-> beta_t  : cos=%+.3f  dot=%+.3f" % cd(beta_v_s, beta_t_s))
print("  beta_v  <-> gamma_v : cos=%+.3f" % (cd(beta_v_s, gamma_v_s)[0]))
print("  beta_t  <-> gamma_t : cos=%+.3f" % (cd(beta_t_s, gamma_t_s)[0]))

# modality gap direction = mean image embedding - mean text embedding (both L2-normalized first)
mi = (final_embeddings_images / final_embeddings_images.norm(dim=-1, keepdim=True)).mean(0)
mt = (final_embeddings_texts  / final_embeddings_texts.norm(dim=-1, keepdim=True)).mean(0)
gap = mi - mt
print("\nmodality-gap alignment (is beta the offset that separates the modalities?):")
print("  beta_v <-> gap : cos=%+.3f" % (cd(beta_v_s, gap)[0]))
print("  beta_t <-> gap : cos=%+.3f" % (cd(beta_t_s, gap)[0]))
print("  beta_v <-> mean_img : cos=%+.3f | beta_t <-> mean_txt : cos=%+.3f"
      % (cd(beta_v_s, mi)[0], cd(beta_t_s, mt)[0]))


cross-encoder (shared space):
  gamma_v <-> gamma_t : cos=-0.105  dot=-33.620
  beta_v  <-> beta_t  : cos=+0.213  dot=+2.919
  beta_v  <-> gamma_v : cos=+0.780
  beta_t  <-> gamma_t : cos=+0.637

modality-gap alignment (is beta the offset that separates the modalities?):
  beta_v <-> gap : cos=+0.048
  beta_t <-> gap : cos=-0.082
  beta_v <-> mean_img : cos=+0.570 | beta_t <-> mean_txt : cos=+0.274


In [ ]:
# --- beta vs the dataset mean of each encoder: is beta the modality gap? ---
# beta_s is added to EVERY embedding, so it is literally an additive term in the dataset mean.
# Text "dataset" = the corpus the text encoder ran on (final_embeddings_texts_decomp).
mean_img = final_embeddings_images.mean(0)                 # vision dataset mean (raw)
mean_txt = final_embeddings_texts_decomp.mean(0)          # text  dataset mean (raw)

def cn(a, b):   # cosine, and ||a||/||b||
    return (a @ b / (a.norm() * b.norm())).item(), (a.norm() / b.norm()).item()

print("beta vs its OWN tower's dataset mean (how much of the mean IS the LN bias):")
c, r = cn(beta_v_s, mean_img); print(f"  vision: cos(beta_v, mean_img) = {c:+.3f}   ||beta||/||mean|| = {r:.2f}")
c, r = cn(beta_t_s, mean_txt); print(f"  text  : cos(beta_t, mean_txt) = {c:+.3f}   ||beta||/||mean|| = {r:.2f}")

gap = mean_img - mean_txt                                  # modality gap (difference of cloud means)
print("\nis beta the modality gap?  gap = mean_img - mean_txt")
print(f"  cos(beta_v,        gap) = {cn(beta_v_s, gap)[0]:+.3f}")
print(f"  cos(beta_t,        gap) = {cn(beta_t_s, gap)[0]:+.3f}")
cg, rg = cn(beta_v_s - beta_t_s, gap)
print(f"  cos(beta_v - beta_t, gap) = {cg:+.3f}   ||beta_v-beta_t||/||gap|| = {rg:.2f}")
print("  -> beta explains the gap iff this cosine ~1 AND the norm ratio ~1 "
      "(else beta is only part of the offset)")


## Stage B — per-sample multiplicative & subtracted blocks (SVD + cosine)
Needs per-sample `a = mu/sqrt(sig^2+eps)` from the final LN. We get it with ONE forward pass using the PRS hooks (which already log `post_ln_mean`/`post_ln_std`), computing `M` and `a` together so they are aligned.

**Verify before trusting:** confirm the recomputed `M` reproduces the residual identity and that `encode_image/encode_text` are called with your model's expected args.

In [24]:
# --- one forward pass to get M and a=mu/std, aligned, for a modest image subset + the text corpus ---
@torch.no_grad()
def ln_stats_vision(indices, batch=64):
    ds_model = ImageNet(root=path + "imagenet/", split="val", transform=preprocess)
    prs = hook_prs_logger(model, device, spatial=False, vision_projection=True)
    Ms, As = [], []
    for s in range(0, len(indices), batch):
        imgs = torch.stack([ds_model[j][0] for j in indices[s:s+batch]]).to(device)
        prs.reinit()
        rep = model.encode_image(imgs, attn_method="head_no_spatial", normalize=False)  # M
        Ms.append(rep.float().cpu())
        As.append((prs.post_ln_mean / prs.post_ln_std).float().cpu())                    # [b,1]
    return torch.cat(Ms), torch.cat(As)

@torch.no_grad()
def ln_stats_text(sentences, batch=256):
    prs = hook_prs_logger_text(model, device, spatial=False, text_projection=True)
    Ms, As = [], []
    for s in range(0, len(sentences), batch):
        toks = tokenizer(list(sentences[s:s+batch])).to(device)
        prs.reinit(); prs.eos_idx = toks.argmax(dim=-1)
        rep = model.encode_text(toks, normalize=False, attn_method="head_no_spatial")                                   # M
        Ms.append(rep.float().cpu())
        As.append((prs.post_ln_mean / prs.post_ln_std).float().cpu())                    # [b,1] EOS-gathered
    return torch.cat(Ms), torch.cat(As)

img_idx = list(ds_vis_.indices)[:2000]                        # subset for speed
M_v, a_v = ln_stats_vision(img_idx)
M_t, a_t = ln_stats_text(corpus_texts)
print("vision:", tuple(M_v.shape), "a", tuple(a_v.shape), "| text:", tuple(M_t.shape), "a", tuple(a_t.shape))


IndexError: index 12 is out of range

In [ ]:
# --- build blocks, SVD, characterize PCs by nearest probe texts ---
def blocks_svd(M, a, gamma_s, beta_s, tag, k=4):
    sub  = a * gamma_s[None, :] - beta_s[None, :]            # [N,d]  = a*gamma - beta
    mult = M + sub                                           # [N,d]  = M + a*gamma - beta
    for blk_name, X in [("mult", mult), ("sub", sub)]:
        Xc = X - X.mean(0)
        U, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
        ev = (S**2 / (S**2).sum())
        r90 = int((ev.cumsum(0) < 0.90).sum().item()) + 1
        cos_g = abs((Vh[0] @ gamma_s / (Vh[0].norm() * gamma_s.norm())).item())
        print(f"\n[{tag} {blk_name}] PC energy {[round(e,3) for e in ev[:k].tolist()]}  r90={r90}  |cos(PC0,gamma)|={cos_g:.2f}")
        for j in range(k):
            print(f"    PC{j}:", [t for t,_ in nearest(Vh[j], txt_bank, texts_str, 4)])
    return mult, sub

mult_v, sub_v = blocks_svd(M_v, a_v, gamma_v_s, beta_v_s, "VISION")
mult_t, sub_t = blocks_svd(M_t, a_t, gamma_t_s, beta_t_s, "TEXT")

# cross-encoder cosine of the multiplicative block PC0
def pc0(X):
    Xc = X - X.mean(0); return torch.linalg.svd(Xc, full_matrices=False)[2][0]
print("\ncross-encoder: mult PC0_v <-> PC0_t cos = %+.3f" % (cd(pc0(mult_v), pc0(mult_t))[0]))



[VISION mult] PC energy [0.127, 0.064, 0.049, 0.038]  r90=131  |cos(PC0,gamma)|=0.17
    PC0: ['An image of a dugong.', 'An image of a leatherback sea turtle.', 'An image of a scaly animal underwater.', 'An image of a large underwater crustacean.']
    PC1: ['An image of a killer whale.', 'An image of a great white shark.', 'An image of a tiger shark.', 'An image of a hammerhead shark.']
    PC2: ['An image of a limpkin.', 'An image of a Gordon Setter.', 'An image of a vulture.', 'An image of an Otterhound.']
    PC3: ['An image of a Japanese Chin.', 'An image of a chicken with feathers and a red comb on its head.', 'An image of a field with chickens pecking for food.', 'An image of a Briard.']

[VISION sub] PC energy [1.0, 0.0, 0.0, 0.0]  r90=1  |cos(PC0,gamma)|=1.00
    PC0: ['An image of a hand pressing a save icon on a computer screen.', 'An image of a professor writing equations on a chalkboard.', 'An image of a city skyline with cranes and buildings under construction.', 'An ima

NameError: name 'M_t' is not defined

In [23]:
# --- WHOLE subtracted block (a*gamma - beta) vs dataset mean / modality gap ---
# The sub block is what LN removes from x each sample; mean(sub) is the true constant offset
# (beta PLUS the a*gamma term). Uses M_v/M_t/sub_v/sub_t from Stage B (same pass -> consistent).
def cn(a, b):
    return (a @ b / (a.norm() * b.norm())).item(), (a.norm() / b.norm()).item()

mean_v, mean_t = M_v.mean(0), M_t.mean(0)          # dataset means (same subset as the blocks)
gapB = mean_v - mean_t
sub_v_mean, sub_t_mean = sub_v.mean(0), sub_t.mean(0)

print("whole sub-block mean vs beta alone (does the a*gamma term add much?):")
print(f"  vision: cos(mean(sub_v), -beta_v) = {cn(sub_v_mean, -beta_v_s)[0]:+.3f}   "
      f"||mean(sub_v)||/||beta_v|| = {sub_v_mean.norm()/beta_v_s.norm():.2f}")
print(f"  text  : cos(mean(sub_t), -beta_t) = {cn(sub_t_mean, -beta_t_s)[0]:+.3f}   "
      f"||mean(sub_t)||/||beta_t|| = {sub_t_mean.norm()/beta_t_s.norm():.2f}")

print("\nsub-block mean vs its own dataset mean:")
print(f"  vision: cos(mean(sub_v), mean_img) = {cn(sub_v_mean, mean_v)[0]:+.3f}")
print(f"  text  : cos(mean(sub_t), mean_txt) = {cn(sub_t_mean, mean_t)[0]:+.3f}")

# sub contribution to the gap: M = mult - sub  ->  the sub term pushes the gap by (mean(sub_t)-mean(sub_v))
print("\nis the subtracted block the modality gap?  (gap = mean_img - mean_txt)")
cs, rs = cn(sub_t_mean - sub_v_mean, gapB)
print(f"  cos(mean(sub_t)-mean(sub_v), gap) = {cs:+.3f}   ||.||/||gap|| = {rs:.2f}")
print("  -> the LN offset explains the gap iff cosine ~1 AND ratio ~1")


whole sub-block mean vs beta alone (does the a*gamma term add much?):
  vision: cos(mean(sub_v), -beta_v) = +1.000   ||mean(sub_v)||/||beta_v|| = 0.98
  text  : cos(mean(sub_t), -beta_t) = +0.836   ||mean(sub_t)||/||beta_t|| = 0.78

sub-block mean vs its own dataset mean:
  vision: cos(mean(sub_v), mean_img) = -0.427
  text  : cos(mean(sub_t), mean_txt) = +0.076

is the subtracted block the modality gap?  (gap = mean_img - mean_txt)
  cos(mean(sub_t)-mean(sub_v), gap) = +0.025   ||.||/||gap|| = 0.61
  -> the LN offset explains the gap iff cosine ~1 AND ratio ~1
